# Day 9 (21 November 2025)

## Control Flow Statements
We now move on to control flow statements. These are the nuts and bolts of an imperative language: `if`, `match`, `while`, `for`, `break`, `continue`, `else (on loops)`, `pass`, `try-raise`, `with`. Function calls also affect the control flow.

### `if`, `if-elif`, `if[-elif]-else`

- You can use any Python expression as the condition in an `if` or `elif` clause (except NumPy arrays, of which I want to find out more, hopefully soon). Using an expression this way is known as using it in a _Boolean context_. In this context, any value is taken as being either true or false. I first read that a value (whose type is not necessarily boolean) that is 'true' in a boolean context is called _truthy_ (otherwise it's 'false' and called _falsy_) in Douglas Crockford's book, _JavaScript: The Good Parts_.
- Thus, like any other dynamic language, Python had to have rules: which values are truthy and which falsy.
- The literal boolean value `True`, any nonzero number or nonempty container (string, tuple, list, dictionary, set, etc.) are truthy, while the literal boolean value `False`, zero (0, of any numeric type), (the literal value) `None`, and empty containers are falsy.
- In boolean contexts, the elegant `if x` is preferred (and correct) to `if x is True`, `if x == True`, and `if bool(x)`.

### The `match` Statement (Python 3.10 Onward)
- I explored "[Pattern Matching](https://en.wikipedia.org/wiki/Pattern_matching)" (not "Pattern Recognition") in programming languages as I read this section. This is an idea inspired by functional languages. I vaguely recalled that I had first heard of such pattern matching in Scala.
- Wikipedia says: In computer science, pattern matching is the act of checking a given sequence of tokens for the presence of the constituents of some pattern. In contrast to pattern recognition, the match usually must be exact: "either it will or will not be a match."
- I then saw that [Java SE 9 has enhanced pattern matching support](https://docs.oracle.com/en/java/javase/21/language/pattern-matching.html) in various contexts. For example, it now supports pattern matching of its `instanceof` operator.
- It lets us do with values of other types what regular expressions let us do with strings: easy testing of the structure and contents of values.
- Warning: **Do not use `match` unless there is a need to analyze the structure of an object.**
- It seems to me that `match` is an attempt to generalize the conventional `switch-case`-like statement (which is a syntactic alternative to long `if-else if-else` blocks). Like the `if` statement imposes a boolean context (in that its expression must evaluate to either `True` or `False`), the `match-case` statement examines if a value matches a pattern. The result of the match is either `True` or `False`.
- The general syntax is:
```
match <expression>:
    case <pattern> [if <guard>]:
        <statements>
# The value of the <expression> is examined for a match a <pattern>. A <pattern> is itself an expression. In other words, the statement is comparing two expressions for a match, rather than equality.
# Remember, use it only to analyze the structure of an object; in Python every value is an object

- My general feeling (based on a first reading of this feature) is that Python offers too much in `match-case`. The point that bears repetition is **the need to analyze the structure of a value**.

#### Kinds of Pattern
I am only going to list all the patterns here, because some of them don't make sense to me (i.e. I am wondering where they are useful). However, I do see how they can lead to succinct code. For now, I am going to remember that whenever I the need to analyze the contents (structure) of a value arises, I am going to refer to this section, rather than having to learn it thoroughly.

##### Literal Pattern
The value literally matches the pattern.
##### Wildcard Pattern
This is simply `_` (the underscore) which matches any value.
##### Capture Pattern
##### Value Pattern
##### OR Pattern
##### Group Pattern
##### Sequence Pattern
Matches a sequence, like, for example, a tuple.
##### `as` Pattern
##### Mapping Pattern
##### Class Pattern


### The `while` Statement
It has what we expect from a `while` statement; except that it has an `else` clause which is somewhat dubious.
```
while <expression>:
    <statement(s)>
```

### The `for` Statement (and Related Ideas)
```
for <target> in <iterable>:
    <statement(s)>
```
Like `while`, `for` can also include an `else` clause.

- `<target>` is the _control variable_ of the loop. Be careful with it because it is bound and rebound only as long as the `<iterable>` has a _next_ element. If it does not have a next element, it is never bound and a `NameError` may occur:

In [35]:
xs = [10, 2, -3]
for x in xs:
    pass # do nothing
print(x) # x is defined and refers to an integer whose value is -3
ys = []
for y in ys:
    pass # do nothing
print(y) # y is undefined!


-3


NameError: name 'y' is not defined

- Note: `x` and `y` were _defined_ inside the `for` loop, and yet they are available outside of it (the `print` statement). This is peculiar as far as the _scope_ of identifiers concerned. I'm sure we'll study "variable scope" later (as it pertains to Python).
- Note: In the above code, what is true of `x` is not true of `y`. The first `for` loop defines `x`, but the second `for` loop does not define `y` because the iterator on `ys` has no elements. Therefore, the `print` statement raises a `NameError`!

#### Iterators
- An iterator is an object `it` such that you can call `next(it)` (note: `next`, like `len` is a global function built into Python), which returns the next item of iterator `it` or, when exhausted, raises a `StopIteration` exception.
- Most iterators are built by implicit or explicit calls to the built-in function `iter`.
- Calling a generator also returns in iterator. We'll learn about generators shortly.
- The `for` loop for iteration is equivalent to the `while` loop:
```
for <x> in <xs>:
    <statement(s)>
    
<it> = iter(<xs>) # this calls the dunder method: class_of_xs.__iter__ method
--
while True:
    try:
        x = next(<it>)
    except StopIteration:
        break
    <statement(s)>
```
- I suddenly remembered Joshua Bloch's advice about handling normal situations using exceptions. He argues in _Effective Java_ that reaching the end of an iterator is not exceptional. But Python philosophy disagrees. When I asked ChatGPT about it, it said:

1) Generality: Works for all iterables, including lazy generators.
2) C-API simplicity: `tp_iternext` returns one object or NULL; no extra "has next" state needed.
3) EAFFPP (Easier to Ask for Following Forgiveness than Prior Permission -- this is my term, ChatGPT used EAFP, which is not as creative ;-)) philosophy: Try and catch, rather than pre-check.
4) Hidden from users: High-level for loops are clean; users never see the exception unless implementing low-level iteration.

It then made the following Unicode diagram:

```
Early Python (pre-2.2)
┌──────────────────────────────┐
│ Index-based loops:           │
│ for i in range(len(xs)):     │
│     x = xs[i]                │
│ No general iterator protocol │
└──────────────────────────────┘
                  │
                  ▼
Python 2.2 (2001)
┌──────────────────────────────┐
│ Introduction of Generators   │
│ def gen():                   │
│     yield ...                │
│ End of generator unknown     │
│ until next() is called       │
└──────────────────────────────┘
                  │
                  ▼
Problem: “How to signal end?”
- Generators: cannot pre-check if next exists
- Builtins: end-of-list must be uniform
                  │
                  ▼
Design Choice: Use Exception
┌──────────────────────────────┐
│ StopIteration                │
│ - next() returns next item   │
│   or raises StopIteration    │
│ - for loop catches it        │
│   automatically              │
│ - Clean C API: tp_iternext() │
│   returns object or NULL     │
└──────────────────────────────┘
                  │
                  ▼
High-level Python usage
┌──────────────────────────────┐
│ for x in xs:                 │
│     print(x)                 │
│ ---------------------------- │
│ No exceptions visible        │
│ StopIteration is internal    │
└──────────────────────────────┘
                  │
                  ▼
Benefits
- Uniform iteration for lists, tuples, generators
- Lazy evaluation possible
- Minimal C-level bookkeeping
- EAFFPP style aligned

```

- I will need to rethink about this. Perhaps it's better to learn the rules like a pro, so that you can break them like an artist.

#### `Iterable`s and `Iterator`s
- Python has made this rather too complex. Here's what we need to remember. An `Iterable`  implements the `__iter__` method which returns a new `Iterator`. Typically, the `for` loop wants an `Iterable`:
```
for <x> in <iterable>:
    # do something with <x>
```
However, the returned `Iterator` also implements the `__iter__` method; so you could use that too in the `for` loop. However, using an `Iterable` in a `for` loop gives a new `Iterator` every time, whereas using an `Iterator` typically returns itself. 

To add to the complexity, there is a global function named `iter` that takes an `Iterable` to produce an `Iterator`. There's also a module named `itertools`. All this is _iteritis_ :-).

```
<xs_iter> = iter(<some iterable>) # remember <xs_iter> is now an Iterator, not Iterable
for <x> in <xs_iter>: # results in a call to __iter__ on the Iterator
    # do something with <x>
```
The following code illustrates this subtle point. Note that `iter(<iterable-or-iterator>)` and `next(<iterator>)` are global functions.

In [ ]:
xs = [11, 55, 165]
it1 = iter(xs)
print(next(it1)) # it1 has advanced

it2 = iter(it1) # it2 starts from the advanced position
print(next(it2))
print(it1 is it2) # True: proof that it1 and it2 are no different!

it3 = iter(xs)
print(next(it3))

`xs` is an `Iterable`. When it is passed to the `iter` function, a new `Iterator` is returned and a fresh iteration of the collection can be iterated on. `it1` is such an Iterator. 

`it2`, on the other hand, is an `Iterator` received by calling `iter` on another `Iterator`. Therefore, `it2` is the _same object_ as `it1` (IOW, these two identifiers refer to the same object). `it3`, because it is obtained from an `iterable`, is a fresh `iterator` which starts from the first member.

#### `range`
- Looping over a sequence of integers is a common task, so Python provides a built-in function `range` (which has two varieties) to generate an `iterable` over integers in a variety of ways. Note: `range` returns an object (a lightweight one). The simplest way to loop `n` times:

In [ ]:
n = 5
for i in range(n):
    print(f"square of {i} = {i*i}")

In [ ]:
for i in range(-1, -6, -1):
    print(i)

#### List Comprehensions

- *Comprehensions are syntactically pleasant*. However, subtle ideas behind them should also be attended to.
- First, the _intended meaning_ of the word _comprehension_. This word has multiple meanings in English. According to the GCIDE (GNU Collaborative International Dictionary of English) v.0.48 it means 1) The act of comprehending, containing, or comprising; inclusion, and 2) **That which is comprehended or enclosed within narrow limits; a summary; an epitome**.
- I believe that 2) is closer in spirit to Python's various _comprehensions_. I checked the original [PEP-202](https://peps.python.org/pep-0202/), but it doesn't expound on it.
- We are literally trying to enclose a list (or a set or a dictionary) _within succinct, narrow limits_ when we use a _comprehension_. The idea is to make a suitable copy of another list or _iterable_. The focus is on a syntax that is both succinct and expressive. It also promotes immutability of collections: if you have a collection, rather than modifying it, construct another collection that makes a (shallow) copy of its members of interest.
- This provides an alternate perspective on operations on collections. Consider filtering out members of a list. One view is to think of filtering out members of the list--an _in-place_ removal operation. That mutates the list. Another view is to construct another list that contains only the required members from the original list leaving it unmodified. 
- Are the two scenarios equivalent? Not internally. But from the programmer's standpoint, they are. It changes the way  programmer implements programs, of course.
- By far the idiomatic way in Python is to use list comprehensions to build new collections from existing ones. Some may say that it is the so-called _syntactic sugar_, but if a declarative syntax makes something more succinct and readable, why not use it? 
- List comprehensions go one step further. With Python's slice assignment, they provide a safe way to mutate an existing list in place.

- The Syntax:
```
[<expression> for <target> in <iterable> <lc-clauses>]
```
where,
`<lc-causes>` are of the form:
- `for <target> in <iterable>`, or
- `if <expression>`


- There's one temptation one should avoid: Just because list comprehensions make the making of new lists very easy doesn't mean we must use them all the time. Making of a collection takes memory and one should be built only when necessary. **When you want to only generate a sequence derived from the members of an existing list, use generators, not list comprehensions.**

Let's analyze some code examples to demonstrate syntactic purity of list comprehensions. I have taken these from the original PEP-202.

<div style="padding: 10px; border-left: 4px solid #888;">

##### Question 1

Create a list of 100 consecutive odd integers greater than or equal to 1:
```
odds = []
for i in range(1, 200, 2):
    odds.append(i)
```
<details>
<summary><strong>Hint: Use List Comprehensions</strong></summary>

```
odds = [x for x in range(1, 200, 2)] # or,
odds = [2 * x + 1 for x in range(100)]
```
</details>

</div>

<div style="padding: 10px; border-left: 4px solid #888;">

##### Question 2

Suppose you have two lists, `xs, ys`. Create a list of 2-tuples: $[(x_i, y_i)] \text{ such that } 0\leq i<m, \text{ where } m = min(len(xs), len(ys))$
<details>
<summary><strong>Hint: Use "zipping" and List Comprehensions</strong></summary>

Without using comprehensions:

```
    m = min(len(xs), len(ys))
    pairs = []
    for i in range(m):
        pairs.append((xs[i], ys[i]))
    
```

We can combine a powerful functional programming construct, [_zipping_](https://en.wikipedia.org/wiki/Zipping_(computer_science)) with list comprehensions:

```
pairs = [p for p in zip(xs, ys)]
```

Advantages of brevity are even more apparent when we have certain conditions. Conditions can be subsumed within additional `<lc-causes>`.
</details>

</div>

Similarly, we have _set_ and _dictionary_ comprehensions.

- A set comprehension has exactly the same syntax and semantics as a list comprehension, except that you enclose it in braces (`{}`) rather than in brackets (`[]`). The result is a set; for example:

```
s = {n//2 for n in range(10)}
print(sorted(s))  # prints: [0, 1, 2, 3, 4]
```
A similar list comprehension would have each item repeated twice, but building a set removes duplicates.

In [36]:
s = {n//2 for n in range(10)}
print(sorted(s))

[0, 1, 2, 3, 4]


- A dictionary comprehension has the same syntax as a set comprehension, except that instead of a single expression before the for clause, you use two expressions with a colon (`:`) between them: key:value. The result is a dict, which retains insertion order. For example, (using the global function `enumerate` that returns an `Iterable` of 2-tuples):

In [39]:
d = {s : i for (i, s) in enumerate(['zero', 'one', 'two'])}
print(d)

{'zero': 0, 'one': 1, 'two': 2}
